In [2]:
import os
import torch
import torch.nn as nn
import json
import re
from torch.utils.data import DataLoader
import torch.optim as optim
from model.LSTM import LSTMModel
from model.RNN import RNNModel
from dataloader.Dataset import AirportDataset
from tqdm import tqdm
import numpy as np

In [3]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===== 配置 =====
dict_path = './airport_dict.json'
file_paths = [
    './sequence_split/202409/202409_3.txt',
]
seq_length = 3
embedding_dim = 36
hidden_dim = 32
num_layers = 5
num_epochs = 30
batch_size = 32
lr = 0.001

In [4]:
# ===== 数据准备 =====
with open(dict_path, 'r', encoding='utf-8') as f:
    airport_to_city = json.load(f)
airport_to_index = {airport: idx for idx, airport in enumerate(airport_to_city)}
index_to_airport = {idx: airport for airport, idx in airport_to_index.items()}
num_classes = len(airport_to_index)
pattern = re.compile(r'\b[A-Z]{3}\b')

dataset = AirportDataset(file_paths, airport_to_index, pattern, seq_length)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_data, val_data = torch.utils.data.random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

Processing ./sequence_split/202409/202409_3.txt: 100%|██████████| 2697603/2697603 [00:12<00:00, 218672.57it/s]


In [4]:
# ===== 训练循环 =====
def train_one_model(model, model_name):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_val_loss = float('inf')
    out_dir = f'./output/{model_name.lower()}'
    os.makedirs(out_dir, exist_ok=True)
    for epoch in range(num_epochs):
        model.train()
        for batch in tqdm(train_loader, desc=f"Training {model_name} Epoch {epoch+1}"):
            batch = batch.to(device)
            inputs = batch[:, :-1]
            targets = batch[:, -1].to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs.view(-1, num_classes), targets.view(-1))
            loss.backward()
            optimizer.step()
        # 验证
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                inputs = batch[:, :-1]
                targets = batch[:, -1].to(device)
                outputs = model(inputs)
                loss = criterion(outputs.view(-1, num_classes), targets.view(-1))
                val_loss += loss.item()
        avg_val_loss = val_loss / len(val_loader)
        print(f'[{model_name}] Epoch {epoch+1}: Training Loss={loss.item():.4f}, Validation Loss={avg_val_loss:.4f}')
        # 保存最优
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), os.path.join(out_dir, f'{model_name.lower()}_seq3_best.pth'))
        torch.save(model.state_dict(), os.path.join(out_dir, f'{model_name.lower()}_seq3_last{epoch+1}.pth'))
    print(f'[{model_name}] Training Done. Best Val Loss: {best_val_loss:.4f}')
    torch.save(model.state_dict(), os.path.join(out_dir, f'{model_name.lower()}_seq3_final.pth'))

In [5]:
# 初始化模型
lstm_model = LSTMModel(num_classes, embedding_dim, hidden_dim, num_layers).to(device)
rnn_model = RNNModel(num_classes, embedding_dim, hidden_dim, num_layers).to(device)

In [6]:
lstm_model.load_state_dict(torch.load(r'.\output\lstm\lstm_seq3_best.pth'))
rnn_model.load_state_dict(torch.load(r'.\output\rnn\rnn_seq3_best.pth'))

<All keys matched successfully>

In [ ]:
# 训练LSTM和RNN
train_one_model(lstm_model, "LSTM")
train_one_model(rnn_model, "RNN")

In [7]:

# ===== 连续1000步预测函数 =====
def continuous_predict(model, seed_seq, airport_to_index, index_to_airport, steps=1000):
    model.eval()
    input_idxs = [airport_to_index[code] for code in seed_seq]
    results = input_idxs.copy()
    with torch.no_grad():
        for _ in range(steps):
            inp = torch.tensor([results[-seq_length:]], dtype=torch.long, device=device)  # shape: (1, seq_length)
            out = model(inp)
            next_idx = torch.argmax(out, dim=1).item()
            results.append(next_idx)
    return [index_to_airport[idx] for idx in results]

In [9]:
# ===== 返回率计算函数 =====
def return_rate(pred_seq, n):
    """
    n=2: 两步返回率。即当前位置与前n步是否相同
    n=3: 三步返回率。
    """
    total = len(pred_seq) - n
    count = sum(1 for i in range(n, len(pred_seq)) if pred_seq[i] == pred_seq[i-n])
    return count / total if total > 0 else 0

In [15]:
# ===== 进行1000步预测并统计 =====
def evaluate_model(model, model_name):
    # 选一组真实起始种子
    # 用你自己的真实序列起始，例如["PEK","SHA","CAN"]，这里举例用数据集第0条
    seed_seq = [index_to_airport[idx.item()] for idx in dataset[3][:seq_length]]
    print(f"[{model_name}] Seed sequence: {seed_seq}")
    pred_seq = continuous_predict(model, seed_seq, airport_to_index, index_to_airport, steps=100)
    two_step = return_rate(pred_seq, 2)
    three_step = return_rate(pred_seq, 3)
    print(f"[{model_name}] 1000步预测 两步返回率: {two_step:.4f}，三步返回率: {three_step:.4f}")
    # 可以保存预测序列
    with open(f'./output/{model_name.lower()}_seq3_1000pred.txt', 'w') as f:
        f.write(' '.join(pred_seq))
    return two_step, three_step
evaluate_model(lstm_model, "LSTM")
evaluate_model(rnn_model, "RNN")

[LSTM] Seed sequence: ['EYW', 'FLL', 'ALB']
[LSTM] 1000步预测 两步返回率: 0.3267，三步返回率: 0.9700
[RNN] Seed sequence: ['EYW', 'FLL', 'ALB']
[RNN] 1000步预测 两步返回率: 0.4851，三步返回率: 0.0000


(0.48514851485148514, 0.0)

In [16]:

# ===== 计算两步返回率和三步返回率 =====
def calculate_return_rates(sequence: list) -> dict:
    """
    计算两步返回率和三步返回率
    :param sequence: 输入序列（list，可以是机场代码字符串列表）
    :return: 字典 { 'two_step_return': float, 'three_step_return': float }
    """
    n = len(sequence)
    if n < 3:
        return {'two_step_return': 0.0, 'three_step_return': 0.0}

    # 两步返回率：隔一个位置相同的数量 / 总数量
    two_step_count = sum(1 for i in range(n - 2) if sequence[i] == sequence[i + 2])
    two_step_rate = two_step_count / (n - 2)

    # 三步返回率：隔两个位置相同的数量 / 总数量
    three_step_count = sum(1 for i in range(n - 3) if sequence[i] == sequence[i + 3])
    three_step_rate = three_step_count / (n - 3)

    return {
        'two_step_return': two_step_rate,
        'three_step_return': three_step_rate
    }

with open("./output/lstm_seq3_1000pred.txt", "r", encoding="utf-8") as f:
    content = f.read()
# 按空格分割
sequence = content.strip().split()
rates = calculate_return_rates(sequence)
print("两步返回率:", rates['two_step_return'])
print("三步返回率:", rates['three_step_return'])

两步返回率: 0.32673267326732675
三步返回率: 0.97


In [22]:

# ===== 计算两步返回率和三步返回率 =====
def calculate_return_rates(sequence: list) -> dict:
    """
    计算两步返回率和三步返回率
    :param sequence: 输入序列（list，可以是机场代码字符串列表）
    :return: 字典 { 'two_step_return': float, 'three_step_return': float }
    """
    n = len(sequence)
    if n < 3:
        return {'two_step_return': 0.0, 'three_step_return': 0.0}

    # 两步返回率：隔一个位置相同的数量 / 总数量
    two_step_count = sum(1 for i in range(n - 2) if sequence[i] == sequence[i + 2])
    two_step_rate = two_step_count / (n - 2)

    # 三步返回率：隔两个位置相同的数量 / 总数量
    three_step_count = sum(1 for i in range(n - 3) if sequence[i] == sequence[i + 3])
    three_step_rate = three_step_count / (n - 3)

    return {
        'two_step_return': two_step_rate,
        'three_step_return': three_step_rate
    }

with open("./output/rnn_seq3_1000pred.txt", "r", encoding="utf-8") as f:
    content = f.read()
# 按空格分割
sequence = content.strip().split()
rates = calculate_return_rates(sequence)
print("两步返回率:", rates['two_step_return'])
print("三步返回率:", rates['three_step_return'])

两步返回率: 0.4995004995004995
三步返回率: 0.0


In [13]:
import plotly.graph_objects as go
from collections import Counter

def read_sequence(filename):
    with open(filename, "r", encoding="utf-8") as f:
        return f.read().strip().split()

def build_sankey_data(sequence):
    links = Counter((sequence[i], sequence[i+1]) for i in range(len(sequence) - 1))
    label_list = list(set(sequence))
    label_map = {k: i for i, k in enumerate(label_list)}
    sources, targets, values = [], [], []
    for (src, tgt), val in links.items():
        sources.append(label_map[src])
        targets.append(label_map[tgt])
        values.append(val)
    return label_list, sources, targets, values

def plot_sankey(label_list, sources, targets, values, title):
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=label_list,
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
        )
    )])
    fig.update_layout(title_text=title, font_size=10)
    fig.show()

# RNN 序列桑基图
rnn_seq = read_sequence("./output/rnn_seq3_1000pred.txt")
label_rnn, src_rnn, tgt_rnn, val_rnn = build_sankey_data(rnn_seq)
plot_sankey(label_rnn, src_rnn, tgt_rnn, val_rnn, "RNN 序列桑基图")

# LSTM 序列桑基图
lstm_seq = read_sequence("./output/lstm_seq3_1000pred.txt")
label_lstm, src_lstm, tgt_lstm, val_lstm = build_sankey_data(lstm_seq)
plot_sankey(label_lstm, src_lstm, tgt_lstm, val_lstm, "LSTM 序列桑基图")

In [5]:
from pyecharts.charts import Sankey
from pyecharts import options as opts
from collections import Counter

def read_sequence(filename):
    with open(filename, "r", encoding="utf-8") as f:
        return f.read().strip().split()

def build_sankey_data(sequence):
    links_counter = Counter((sequence[i], sequence[i+1]) for i in range(len(sequence)-1))
    nodes = [{"name": s} for s in set(sequence)]
    links = [{"source": src, "target": tgt, "value": val} for (src, tgt), val in links_counter.items()]
    return nodes, links

def plot_sankey_pyecharts(nodes, links, title="桑基图"):
    sankey = (
        Sankey()
        .add(
            "机场流转",
            nodes,
            links,
            linestyle_opt=opts.LineStyleOpts(opacity=0.3, curve=0.5),
            label_opts=opts.LabelOpts(position="right"),
        )
        .set_global_opts(title_opts=opts.TitleOpts(title=title))
    )
    return sankey

# RNN
rnn_seq = read_sequence("./output/rnn_seq3_1000pred.txt")
nodes_rnn, links_rnn = build_sankey_data(rnn_seq)
print(nodes_rnn)    # 检查节点是否正确
print(links_rnn)    # 检查连接是否正确
sankey_rnn = plot_sankey_pyecharts(nodes_rnn, links_rnn, "RNN 序列桑基图")
sankey_rnn.render_notebook()  # 显示桑基图

# LSTM
lstm_seq = read_sequence("./output/lstm_seq3_1000pred.txt")
nodes_lstm, links_lstm = build_sankey_data(lstm_seq)
print(nodes_lstm)
print(links_lstm)
sankey_lstm = plot_sankey_pyecharts(nodes_lstm, links_lstm, "LSTM 序列桑基图")
sankey_lstm.render_notebook()

[{'name': 'DEN'}, {'name': 'LAX'}, {'name': 'FLL'}, {'name': 'ATL'}, {'name': 'ALB'}, {'name': 'DFW'}, {'name': 'EYW'}]
[{'source': 'EYW', 'target': 'FLL', 'value': 1}, {'source': 'FLL', 'target': 'ALB', 'value': 1}, {'source': 'ALB', 'target': 'ATL', 'value': 1}, {'source': 'ATL', 'target': 'DEN', 'value': 72}, {'source': 'DEN', 'target': 'ATL', 'value': 143}, {'source': 'ATL', 'target': 'DFW', 'value': 144}, {'source': 'DFW', 'target': 'ATL', 'value': 72}, {'source': 'DFW', 'target': 'DEN', 'value': 142}, {'source': 'DEN', 'target': 'DFW', 'value': 142}, {'source': 'DFW', 'target': 'LAX', 'value': 71}, {'source': 'LAX', 'target': 'ATL', 'value': 71}, {'source': 'ATL', 'target': 'LAX', 'value': 71}, {'source': 'LAX', 'target': 'DEN', 'value': 71}]
[{'name': 'DEN'}, {'name': 'FLL'}, {'name': 'ALB'}, {'name': 'PHX'}, {'name': 'EYW'}]
[{'source': 'EYW', 'target': 'FLL', 'value': 1}, {'source': 'FLL', 'target': 'ALB', 'value': 1}, {'source': 'ALB', 'target': 'DEN', 'value': 1}, {'source':

In [4]:
from collections import Counter

def read_sequence(filename):
    with open(filename, "r", encoding="utf-8") as f:
        return f.read().strip()
    
s = read_sequence("./output/rnn_seq3_1000pred.txt")
words = s.split()

# 统计所有三元组
triplets = [(words[i], words[i+1], words[i+2]) for i in range(len(words) - 2)]

# 统计出现次数
counter = Counter(triplets)

# 打印结果
for triplet, count in counter.items():
    print(f"{triplet}: {count}")

('EYW', 'FLL', 'ALB'): 1
('FLL', 'ALB', 'ATL'): 1
('ALB', 'ATL', 'DEN'): 1
('ATL', 'DEN', 'ATL'): 72
('DEN', 'ATL', 'DFW'): 72
('ATL', 'DFW', 'ATL'): 72
('DFW', 'ATL', 'DFW'): 72
('ATL', 'DFW', 'DEN'): 71
('DFW', 'DEN', 'DFW'): 71
('DEN', 'DFW', 'LAX'): 71
('DFW', 'LAX', 'ATL'): 71
('LAX', 'ATL', 'LAX'): 71
('ATL', 'LAX', 'DEN'): 71
('LAX', 'DEN', 'DFW'): 71
('DEN', 'DFW', 'DEN'): 71
('DFW', 'DEN', 'ATL'): 71
('DEN', 'ATL', 'DEN'): 71


从此处分割，下面为不同路径经过den的预测。结果存储在.\output\lstm_after_den.txt
.\output\rnn_after_den.txt

In [22]:
from tqdm.notebook import tqdm

def predict_next_after_den_unique(model, dataset, index_to_airport, airport_to_index, seq_length, model_name, target_airport="DEN"):
    # 1. 只取唯一的定长seed序列，且最后一个是DEN
    seen = set()
    seed_seqs = []
    for data in dataset:
        seq = tuple(index_to_airport[idx.item()] for idx in data[:seq_length])
        if seq[-1] != target_airport:
            continue
        if seq not in seen:
            seen.add(seq)
            seed_seqs.append(list(seq))

    # 2. 只需一步预测
    results = []
    for seed in tqdm(seed_seqs, desc=f"{model_name} predicting"):
        pred_next = continuous_predict(model, seed, airport_to_index, index_to_airport, steps=1)[-1]
        out_line = ' '.join(seed + [pred_next])
        print(out_line)
        results.append(out_line)

    # 3. 保存为txt
    with open(f'./output/{model_name.lower()}_after_{target_airport.lower()}.txt', "w") as f:
        for line in results:
            f.write(line + "\n")
    return results

# 用法示例
predict_next_after_den_unique(
    lstm_model, dataset, index_to_airport, airport_to_index, seq_length, "LSTM", target_airport="DEN"
)
predict_next_after_den_unique(
    rnn_model, dataset, index_to_airport, airport_to_index, seq_length, "RNN", target_airport="DEN"
)

EYW FLL DEN DEN
EYW MCO DEN DEN
EYW TPA DEN ATL
FLL TPA DEN PHX
MCO FLL DEN MCO
STT SJU DEN ATL
STX SJU DEN ATL
TLH TPA DEN ATL
ABE ATL DEN ATL
AEX ATL DEN DEN
AGS ATL DEN ATL
ALB DTW DEN DEN
ALB LGA DEN DEN
AVL LGA DEN DEN
BDL DTW DEN ATL
BDL LGA DEN ATL
BGM DTW DEN DEN
BGR JFK DEN DEN
BGR LGA DEN DEN
BMI ATL DEN DEN
BTV DTW DEN DEN
BTV JFK DEN DEN
BTV LGA DEN DEN
BUF DTW DEN DEN
BUF JFK DEN DEN
BUF LGA DEN DEN
CAE ATL DEN ATL
CAE LGA DEN DEN
CHA ATL DEN DEN
CHO ATL DEN ATL
CHO LGA DEN ATL
CHS DTW DEN DEN
CID MSP DEN ATL
CLE DTW DEN DEN
CMH MSP DEN ATL
CSG ATL DEN ATL
CVG DTW DEN ATL
CVG MSP DEN EWR
CWA MSP DEN ATL
DHN ATL DEN DEN
DLH MSP DEN DEN
DSM MSP DEN DEN
EVV ATL DEN DEN
FAY ATL DEN ATL
GSO DTW DEN DEN
GSP DTW DEN CLT
GSP LGA DEN ATL
GTR ATL DEN ATL
HPN ATL DEN HPN
ILM ATL DEN DEN
ILM LGA DEN ATL
IND DTW DEN DEN
ITH JFK DEN PHX
JAX LGA DEN DEN
JFK DTW DEN DEN
JFK IND DEN DEN
MCI LGA DEN DEN
MGM ATL DEN EWR
MKE MSP DEN DEN
MLI ATL DEN DEN
MLU ATL DEN DEN
MOB ATL DEN DEN
MSN DTW 

EYW FLL DEN ATL
EYW MCO DEN ATL
EYW TPA DEN ATL
FLL TPA DEN DEN
MCO FLL DEN DFW
STT SJU DEN ORD
STX SJU DEN ATL
TLH TPA DEN ATL
ABE ATL DEN ATL
AEX ATL DEN ATL
AGS ATL DEN ATL
ALB DTW DEN ATL
ALB LGA DEN ATL
AVL LGA DEN DFW
BDL DTW DEN DEN
BDL LGA DEN DEN
BGM DTW DEN DEN
BGR JFK DEN ATL
BGR LGA DEN ATL
BMI ATL DEN ATL
BTV DTW DEN DEN
BTV JFK DEN DEN
BTV LGA DEN ORD
BUF DTW DEN ATL
BUF JFK DEN ATL
BUF LGA DEN ATL
CAE ATL DEN ATL
CAE LGA DEN ATL
CHA ATL DEN ATL
CHO ATL DEN DFW
CHO LGA DEN DFW
CHS DTW DEN ATL
CID MSP DEN DFW
CLE DTW DEN ORD
CMH MSP DEN ATL
CSG ATL DEN DEN
CVG DTW DEN DFW
CVG MSP DEN DFW
CWA MSP DEN ATL
DHN ATL DEN DFW
DLH MSP DEN ATL
DSM MSP DEN DFW
EVV ATL DEN DFW
FAY ATL DEN ORD
GSO DTW DEN DEN
GSP DTW DEN ATL
GSP LGA DEN ORD
GTR ATL DEN ORD
HPN ATL DEN ATL
ILM ATL DEN ATL
ILM LGA DEN ATL
IND DTW DEN DFW
ITH JFK DEN ORD
JAX LGA DEN ATL
JFK DTW DEN DFW
JFK IND DEN DFW
MCI LGA DEN ATL
MGM ATL DEN DFW
MKE MSP DEN ATL
MLI ATL DEN DFW
MLU ATL DEN ATL
MOB ATL DEN DFW
MSN DTW 

['EYW FLL DEN ATL',
 'EYW MCO DEN ATL',
 'EYW TPA DEN ATL',
 'FLL TPA DEN DEN',
 'MCO FLL DEN DFW',
 'STT SJU DEN ORD',
 'STX SJU DEN ATL',
 'TLH TPA DEN ATL',
 'ABE ATL DEN ATL',
 'AEX ATL DEN ATL',
 'AGS ATL DEN ATL',
 'ALB DTW DEN ATL',
 'ALB LGA DEN ATL',
 'AVL LGA DEN DFW',
 'BDL DTW DEN DEN',
 'BDL LGA DEN DEN',
 'BGM DTW DEN DEN',
 'BGR JFK DEN ATL',
 'BGR LGA DEN ATL',
 'BMI ATL DEN ATL',
 'BTV DTW DEN DEN',
 'BTV JFK DEN DEN',
 'BTV LGA DEN ORD',
 'BUF DTW DEN ATL',
 'BUF JFK DEN ATL',
 'BUF LGA DEN ATL',
 'CAE ATL DEN ATL',
 'CAE LGA DEN ATL',
 'CHA ATL DEN ATL',
 'CHO ATL DEN DFW',
 'CHO LGA DEN DFW',
 'CHS DTW DEN ATL',
 'CID MSP DEN DFW',
 'CLE DTW DEN ORD',
 'CMH MSP DEN ATL',
 'CSG ATL DEN DEN',
 'CVG DTW DEN DFW',
 'CVG MSP DEN DFW',
 'CWA MSP DEN ATL',
 'DHN ATL DEN DFW',
 'DLH MSP DEN ATL',
 'DSM MSP DEN DFW',
 'EVV ATL DEN DFW',
 'FAY ATL DEN ORD',
 'GSO DTW DEN DEN',
 'GSP DTW DEN ATL',
 'GSP LGA DEN ORD',
 'GTR ATL DEN ORD',
 'HPN ATL DEN ATL',
 'ILM ATL DEN ATL',


统计后三个序列的出现频率

In [36]:
from collections import Counter

filename = r'.\output\lstm_after_den.txt'  # 替换为你的文件名
# filename = r'.\output\rnn_after_den.txt'  # 替换为你的文件名


counter = Counter()

with open(filename, 'r') as f:
    for line in f:
        codes = line.strip().split()
        if len(codes) >= 3:
            last3 = tuple(codes[-3:])
            counter[last3] += 1

# 取出现次数最多的7个组合
most_common7 = counter.most_common(10)

print("出现次数最多的7个后三位组合和次数：")
for combo, count in most_common7:
    print(' '.join(combo), count)

出现次数最多的7个后三位组合和次数：
DFW DEN ATL 73
ORD DEN DEN 59
ATL DEN DEN 57
DFW DEN DEN 51
ORD DEN ATL 50
CLT DEN ATL 47
DTW DEN DEN 44
ATL DEN ATL 39
MSP DEN DEN 38
PHX DEN DEN 34


In [ ]:
from pyecharts.charts import Sankey
from pyecharts import options as opts

# lines = [
#     "DFW DEN ATL 57",
#     "DFW DEN DFW 51",
#     "ATL DEN ATL 47",
#     "ORD DEN ATL 47",
#     "CLT DEN ATL 41",
#     "ORD DEN DFW 39",
#     "ATL DEN DFW 36",
#     "MSP DEN ATL 33",
#     "CLT DEN DFW 33",
#     "DFW DEN DEN 32",
# ]

lines = [
    "DFW DEN ATL 73",
    "ORD DEN DEN 59",
    "ATL DEN DEN 57",
    "DFW DEN DEN 51",
    "ORD DEN ATL 50",
    "CLT DEN ATL 47",
    "DTW DEN DEN 44",
    "ATL DEN ATL 39",
    "MSP DEN DEN 38",
    "PHX DEN DEN 34",
]

def sankey_pathwise(lines, outfile="sankey_pathwise.html"):
    nodes_set = set()
    links = []

    for line in lines:
        a, b, c, w = line.strip().split()
        w = int(w)
        a1, b2, c3 = a + "1", b + "2", c + "3"
        nodes_set.update([a1, b2, c3])
        links.append({"source": a1, "target": b2, "value": w})
        links.append({"source": b2, "target": c3, "value": w})

    nodes = [{"name": n} for n in nodes_set]

    sankey = (
        Sankey()
        .add(
            "机场三联流向",
            nodes,
            links,
            linestyle_opt=opts.LineStyleOpts(opacity=0.4, curve=0.5, color="source"),
            label_opts=opts.LabelOpts(position="right"),
        )
        .set_global_opts(title_opts=opts.TitleOpts(title="三层机场流向桑基图（路径区分）"))
    )
    sankey.render(outfile)
    print(f"桑基图已生成: {outfile}")

sankey_pathwise(lines, outfile="sankey_pathwise.html")

桑基图已生成: sankey_pathwise.html


单一桑吉图

In [ ]:
from pyecharts.charts import Sankey
from pyecharts import options as opts

# lines = [
#     "DFW DEN ATL 57",
#     "DFW DEN DFW 51",
#     "ATL DEN ATL 47",
#     "ORD DEN ATL 47",
#     "CLT DEN ATL 41",
#     "ORD DEN DFW 39",
#     "ATL DEN DFW 36",
#     "MSP DEN ATL 33",
#     "CLT DEN DFW 33",
#     "DFW DEN DEN 32",
# ]

lines = [
    "DFW DEN ATL 73",
    "ORD DEN DEN 59",
    "ATL DEN DEN 57",
    "DFW DEN DEN 51",
    "ORD DEN ATL 50",
    "CLT DEN ATL 47",
    "DTW DEN DEN 44",
    "ATL DEN ATL 39",
    "MSP DEN DEN 38",
    "PHX DEN DEN 34",
]

def sankey_1to3_colored(lines, outfile="sankey_1to3_colored.html"):
    nodes_set = set()
    links = []

    for line in lines:
        a, _, c, w = line.strip().split()
        w = int(w)
        a1 = a + "1"
        c3 = c + "3"
        nodes_set.update([a1, c3])
        # 默认颜色（可为None或"source"）
        link = {"source": a1, "target": c3, "value": w}
        if a == c:
            # 特殊颜色，比如红色
            link["lineStyle"] = {"color": "yellow", "opacity":0.8}
        links.append(link)

    nodes = [{"name": n} for n in nodes_set]

    sankey = (
        Sankey()
        .add(
            "机场一跳流向",
            nodes,
            links,
            linestyle_opt=opts.LineStyleOpts(opacity=0.5, curve=0.5, color="source"),
            label_opts=opts.LabelOpts(position="right"),
        )
        .set_global_opts(title_opts=opts.TitleOpts(title="机场一跳桑基图（自环特殊标色）"))
    )
    sankey.render(outfile)
    print(f"桑基图已生成: {outfile}")

sankey_1to3_colored(lines, outfile="sankey_1to3_colored.html")

桑基图已生成: sankey_1to3_colored.html


合并的桑吉图

In [43]:
from pyecharts.charts import Sankey
from pyecharts import options as opts

# 柔和科研色
color_palette = [
    "#4E79A7", # 蓝
    "#F28E2B", # 橙
    "#59A14F", # 绿
    "#B07AA1", # 紫
    "#9C755F", # 棕
    "#76B7B2", # 青
    "#FFBE7D", # 浅橙
    "#BAB0AC"  # 灰
]
selfloop_color = "#D62728"  # 非常醒目的红色

lines1 = [
    "DFW DEN ATL 57",
    "DFW DEN DFW 51",
    "ATL DEN ATL 47",
    "ORD DEN ATL 47",
    "CLT DEN ATL 41",
    "ORD DEN DFW 39",
    "ATL DEN DFW 36",
    "MSP DEN ATL 33",
    "CLT DEN DFW 33",
    "DFW DEN DEN 32",
]

lines2 = [
    "DFW DEN ATL 73",
    "ORD DEN DEN 59",
    "ATL DEN DEN 57",
    "DFW DEN DEN 51",
    "ORD DEN ATL 50",
    "CLT DEN ATL 47",
    "DTW DEN DEN 44",
    "ATL DEN ATL 39",
    "MSP DEN DEN 38",
    "PHX DEN DEN 34",
]

def sankey_1to3_colored(lines, title, color_palette, selfloop_color, chart_width='400px', chart_height='500px'):
    nodes_set = set()
    links = []
    color_map = {}
    palette_len = len(color_palette)

    # 为节点分配颜色
    node_bases = []
    for line in lines:
        a, _, c, _ = line.strip().split()
        if a not in node_bases:
            node_bases.append(a)
        if c not in node_bases:
            node_bases.append(c)
    for idx, node in enumerate(node_bases):
        color_map[node] = color_palette[idx % palette_len]

    for line in lines:
        a, _, c, w = line.strip().split()
        w = int(w)
        a1 = a + "1"
        c3 = c + "3"
        nodes_set.update([a1, c3])
        link = {"source": a1, "target": c3, "value": w}
        if a == c:
            link["lineStyle"] = {"color": selfloop_color, "width": 7, "opacity": 0.95}
        else:
            link["lineStyle"] = {"color": color_map[a], "width": 3, "opacity": 0.65}
        links.append(link)

    # 节点着色
    nodes = []
    for n in nodes_set:
        base = n[:-1]
        color = color_map.get(base, "#bab0ac")
        nodes.append({"name": n, "itemStyle": {"color": color}})

    sankey = (
        Sankey(init_opts=opts.InitOpts(width=chart_width, height=chart_height, bg_color="#fff"))
        .add(
            "",
            nodes,
            links,
            linestyle_opt=opts.LineStyleOpts(curve=0.5),
            label_opts=opts.LabelOpts(position="right", font_size=13),
        )
        .set_global_opts(title_opts=opts.TitleOpts(title=title, pos_left="center", title_textstyle_opts=opts.TextStyleOpts(font_size=17)))
    )
    return sankey

# 生成两个图对象
sankey_lstm = sankey_1to3_colored(
    lines1, "LSTM", color_palette, selfloop_color, chart_width='400px', chart_height='500px'
)
sankey_rnn = sankey_1to3_colored(
    lines2, "RNN", color_palette, selfloop_color, chart_width='400px', chart_height='500px'
)

# 分别渲染为HTML字符串
lstm_html = sankey_lstm.render_embed()
rnn_html = sankey_rnn.render_embed()

# 拼接成极简无色块的HTML
html_str = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>LSTM vs RNN Sankey</title>
    <style>
        html,body {{
            background: #fff;
            margin: 0;
            padding: 0;
            font-family: "Helvetica Neue", Arial, sans-serif;
        }}
        .container {{
            display: flex;
            flex-direction: row;
            justify-content: center;
            align-items: flex-start;
            width: 100vw;
            min-height: 100vh;
            background: #fff;
        }}
        .chart {{
            width: 400px;
            margin: 0 20px;
            background: #fff;
        }}
        /* 标题直接在图上方 */
        .chart .chart-title {{
            text-align: center;
            font-size: 1.25em;
            margin-top: 0.6em;
            margin-bottom: 0.1em;
            font-weight: bold;
            letter-spacing: 1px;
            background: #fff;
        }}
        @media (max-width: 900px) {{
            .container {{ flex-direction: column; align-items: center; }}
            .chart {{ margin: 12px auto; }}
        }}
    </style>
</head>
<body>
    <div class="container">
        <div class="chart">
            {lstm_html}
        </div>
        <div class="chart">
            {rnn_html}
        </div>
    </div>
</body>
</html>
"""

with open("compare_sankey.html", "w", encoding="utf-8") as f:
    f.write(html_str)

print("已生成 compare_sankey.html，极简无色块，自环线醒目，科研配色，两图并排！")

已生成 compare_sankey.html，极简无色块，自环线醒目，科研配色，两图并排！
